# 🏭 Fábrica de reels — Iago Rodrigues

Roteiro em texto → sua voz (clonada) → sua boca sincronizada no vídeo-base → legenda → reel 1080x1920 pronto.
Tudo com modelos de código aberto de uso comercial livre (Chatterbox MIT · LatentSync Apache 2.0 · Whisper MIT).

## Como usar (Colab)
1. **Runtime → Change runtime type → T4 GPU** (grátis).
2. No seu Google Drive crie a pasta `fabrica/` com:
   - `voz_ref.wav` — 20–60 s da sua voz, lugar silencioso (o vídeo 2 do briefing serve: o notebook extrai o áudio se você colocar `voz_ref.mp4`)
   - `base/` — os vídeos-base (celular vertical 1080p, olhando pra câmera, sem legenda)
   - `roteiros/` — um `.txt` por reel: 1ª linha `GANCHO: ...`, resto é a fala
3. **Runtime → Run all**. Os reels prontos aparecem em `fabrica/saida/`.

Reels já feitos não são refeitos: pra regerar um, apague a pasta dele em `saida/`.

## Kaggle
Anexe um Dataset privado com a mesma estrutura (`voz_ref.wav`, `base/`, `roteiros/`), ligue **GPU T4** em Settings → Accelerator e rode tudo. A saída fica em `/kaggle/working/saida/` (baixe o zip no fim).


In [ ]:
#@title 1. Onde estão os arquivos
import os, sys, shutil, subprocess, json, time
from pathlib import Path

NO_KAGGLE = Path("/kaggle").exists()
if NO_KAGGLE:
    entradas = [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]
    ENTRADA = next((p for p in entradas if (p / "roteiros").exists()), entradas[0] if entradas else Path("/kaggle/input"))
    SAIDA = Path("/kaggle/working/saida")
    TRABALHO = Path("/kaggle/working/fabrica")
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ENTRADA = Path("/content/drive/MyDrive/fabrica")
    SAIDA = ENTRADA / "saida"
    TRABALHO = Path("/content/fabrica")

SAIDA.mkdir(parents=True, exist_ok=True); TRABALHO.mkdir(parents=True, exist_ok=True)
roteiros = sorted((ENTRADA / "roteiros").glob("*.txt"))
bases = sorted((ENTRADA / "base").glob("*.mp4")) + sorted((ENTRADA / "base").glob("*.mov"))
voz_ref = ENTRADA / "voz_ref.wav"
assert roteiros, f"nenhum roteiro .txt em {ENTRADA/'roteiros'}"
assert bases, f"nenhum vídeo-base em {ENTRADA/'base'}"
print(f"{len(roteiros)} roteiros · {len(bases)} vídeos-base · voz_ref: {'ok' if voz_ref.exists() else 'FALTA (vou tentar extrair de voz_ref.mp4)'}")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "SEM GPU — ative a T4 em Runtime → Change runtime type")


In [ ]:
#@title 2. Instalar (≈4 min na primeira vez)
%%bash
set -e
pip -q install "diffusers==0.32.2" "transformers==4.48.0" accelerate einops omegaconf opencv-python \
  python_speech_features librosa face-alignment kornia DeepCache decord imageio imageio-ffmpeg \
  faster-whisper huggingface_hub soundfile scipy matplotlib ffmpeg-python 2>&1 | tail -1
# Chatterbox pina outro transformers → venv separado, só pra voz
python -m venv /content/venv_tts 2>/dev/null || python -m venv /kaggle/working/venv_tts
VENV=$( [ -d /content/venv_tts ] && echo /content/venv_tts || echo /kaggle/working/venv_tts )
$VENV/bin/pip -q install chatterbox-tts 2>&1 | tail -1
which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu-core)
fc-list | grep -qi dejavu || apt-get -qq install -y fonts-dejavu-core
echo instalado


In [ ]:
#@title 3. LatentSync 1.6 + pesos + patch (detector sem insightface)
LS = TRABALHO / "LatentSync"
if not LS.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/bytedance/LatentSync.git", str(LS)], check=True)
from huggingface_hub import hf_hub_download
for f in ["latentsync_unet.pt", "whisper/tiny.pt"]:
    hf_hub_download("ByteDance/LatentSync-1.6", f, local_dir=str(LS / "checkpoints"))
(LS / "latentsync/utils/face_detector.py").write_text(DETECTOR := r'''# Uso comercial (Mac e nuvem): o insightface (modelos só p/ pesquisa, onnx CUDA) foi
# trocado pela face_alignment (FAN, BSD). O pipeline só consome 3 pontos do
# detector — centro das duas sobrancelhas e centro do nariz — via índices do
# layout de 106 pontos do insightface. Devolvemos um array 106x2 só com esses
# índices preenchidos a partir dos 68 pontos iBUG.
import numpy as np
import torch
import face_alignment as _fa

INSIGHTFACE_DETECT_SIZE = 512


class FaceDetector:
    def __init__(self, device="cuda"):
        dev = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
        self.fan = _fa.FaceAlignment(_fa.LandmarksType.TWO_D, flip_input=False, device=dev)
        self._ultimo = (None, None)  # frame borrado (corte, movimento) reaproveita a detecção anterior

    def __call__(self, frame, threshold=0.5):
        f_h, f_w, _ = frame.shape
        lms = self.fan.get_landmarks(frame[:, :, ::-1])  # FAN espera RGB; o pipeline passa BGR
        if not lms:
            lms = self.fan.get_landmarks(frame)  # tenta na outra ordem de canal antes de desistir
        if not lms:
            return self._ultimo
        lm = max(lms, key=lambda l: (l[:, 0].max() - l[:, 0].min()) * (l[:, 1].max() - l[:, 1].min()))
        lm = np.round(lm).astype(np.int_)

        # iBUG 68: 17-21 sobrancelha esquerda (da imagem), 22-26 direita, 30-33 nariz (ponta/base)
        lmk = np.zeros((106, 2), dtype=np.int_)
        lmk[[43, 48, 49, 51, 50]] = lm[17:22]
        lmk[101:106] = lm[22:27]
        lmk[[74, 77, 83, 86]] = lm[[30, 31, 33, 35]]
        lmk[73] = lm[29]

        x1, y1 = lm[:, 0].min(), lm[:, 1].min()
        x2, y2 = lm[:, 0].max(), lm[:, 1].max()
        y2 += int((x2 - x1) * 0.1)
        x1 -= int((x2 - x1) * 0.05)
        x2 += int((x2 - x1) * 0.05)
        self._ultimo = ((max(0, x1), max(0, y1), min(f_w, x2), min(f_h, y2)), lmk)
        return self._ultimo
''')
print("LatentSync pronto em", LS)


In [ ]:
#@title 4. Código da fábrica (montagem + voz)
(TRABALHO / "montagem.py").write_text(r'''"""Montagem final do reel: legenda palavra a palavra + gancho + 1080x1920.

Compartilhado entre o Mac (reel.py) e o notebook da nuvem. Só depende de
ffmpeg e de um Whisper com timestamp de palavra (mlx-whisper no Mac,
faster-whisper em GPU NVIDIA).
"""
import json, shutil, subprocess, textwrap
from pathlib import Path


def _ffprobe(v: Path, campos: str) -> str:
    return subprocess.run(["ffprobe", "-v", "quiet", "-show_entries", campos, "-of", "csv=p=0", str(v)],
                          capture_output=True, text=True).stdout.strip()


def palavras(wav: Path) -> list:
    """[{word,start,end}] — tenta mlx-whisper (Mac) e cai pra faster-whisper (CUDA/CPU)."""
    try:
        import mlx_whisper
        r = mlx_whisper.transcribe(str(wav), path_or_hf_repo="mlx-community/whisper-large-v3-turbo",
                                   language="pt", word_timestamps=True)
        return [w for s in r["segments"] for w in s.get("words", [])]
    except ImportError:
        pass
    from faster_whisper import WhisperModel
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = WhisperModel("large-v3-turbo", device=dev, compute_type="float16" if dev == "cuda" else "int8")
    segs, _ = m.transcribe(str(wav), language="pt", word_timestamps=True)
    return [{"word": w.word, "start": w.start, "end": w.end} for s in segs for w in (s.words or [])]


def _esc(t: str) -> str:
    return t.replace("\\", "\\\\").replace("{", "\\{").replace("}", "\\}")


def legendas_ass(wav: Path, destino: Path) -> Path:
    """Blocos de 3 palavras, a atual em amarelo. Fonte grande, centro-baixo."""
    ps = palavras(wav)
    (destino / "palavras.json").write_text(json.dumps(ps, ensure_ascii=False))
    cab = textwrap.dedent("""\
        [Script Info]
        ScriptType: v4.00+
        PlayResX: 1080
        PlayResY: 1920

        [V4+ Styles]
        Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
        Style: Leg,DejaVu Sans,86,&H00FFFFFF,&H00FFFFFF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,6,2,2,60,60,300,1

        [Events]
        Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
        """)

    def ts(s):
        return f"{int(s // 3600)}:{int(s % 3600 // 60):02d}:{s % 60:05.2f}"

    linhas = []
    for i in range(0, len(ps), 3):
        grupo = ps[i:i + 3]
        for j, w in enumerate(grupo):
            fim = grupo[j + 1]["start"] if j + 1 < len(grupo) else w["end"]
            txt = " ".join(
                ("{\\c&H00D7FF&}" + _esc(x["word"].strip().upper()) + "{\\c&HFFFFFF&}") if k == j
                else _esc(x["word"].strip().upper())
                for k, x in enumerate(grupo))
            linhas.append(f"Dialogue: 0,{ts(w['start'])},{ts(fim)},Leg,,0,0,0,,{txt}")
    ass = destino / "legenda.ass"
    ass.write_text(cab + "\n".join(linhas) + "\n", encoding="utf-8")
    return ass


def montar(sync: Path, wav: Path, gancho: str, destino: Path, nome: str = "reel_final.mp4") -> Path:
    """sync = vídeo já com a boca sincronizada (qualquer resolução); wav = a voz."""
    destino.mkdir(parents=True, exist_ok=True)
    ass = legendas_ass(wav, destino)
    w, h = (int(x) for x in _ffprobe(sync, "stream=width,height").split("\n")[0].split(","))

    # já é 9:16 (gravação-base no celular) → só escala; senão centraliza num
    # quadro escuro 1080x1920.
    if abs(w / h - 9 / 16) < 0.02:
        filtros = ["scale=1080:1920"]
    else:
        filtros = ["scale=1080:-2", "pad=1080:1920:(ow-iw)/2:(oh-ih)/2:color=#0f1720"]

    if gancho:
        gtxt = destino / "gancho.txt"
        gtxt.write_text("\n".join(textwrap.wrap(gancho.replace("'", "’"), 22)))
        filtros.append(f"drawtext=textfile='{gtxt}':fontsize=72:fontcolor=white:line_spacing=12"
                       f":x=(w-text_w)/2:y=110:box=1:boxcolor=black@0.55:boxborderw=28")
    filtros.append(f"subtitles='{ass}'")

    final = destino / nome
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(sync), "-i", str(wav), "-map", "0:v", "-map", "1:a",
                    "-vf", ",".join(filtros), "-c:v", "libx264", "-preset", "medium", "-crf", "20",
                    "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "160k", "-shortest",
                    "-movflags", "+faststart", str(final)], check=True)
    return final


def ler_roteiro(caminho: Path):
    """1ª linha `GANCHO: ...`; resto é a fala, quebrada em pedaços de ~280 chars."""
    import re
    linhas = caminho.read_text(encoding="utf-8").strip().splitlines()
    gancho = ""
    if linhas and linhas[0].upper().startswith("GANCHO:"):
        gancho = linhas.pop(0).split(":", 1)[1].strip()
    corpo = " ".join(l.strip() for l in linhas if l.strip())
    partes, atual = [], ""
    for frase in re.split(r"(?<=[.!?])\s+", corpo):
        if len(atual) + len(frase) > 280 and atual:
            partes.append(atual.strip()); atual = ""
        atual += " " + frase
    if atual.strip():
        partes.append(atual.strip())
    return gancho, partes
''')
(TRABALHO / "tts.py").write_text(r'''"""Voz clonada do Iago (Chatterbox multilingual, MIT). Roda em venv próprio
porque o chatterbox pina outra versão do transformers que o LatentSync.

    python tts.py <roteiro.txt> <voz_ref.wav> <saida.wav> [--exag 0.5 --cfg 0.5]
"""
import argparse, sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from montagem import ler_roteiro


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("roteiro"); ap.add_argument("voz_ref"); ap.add_argument("saida")
    ap.add_argument("--exag", type=float, default=0.5)
    ap.add_argument("--cfg", type=float, default=0.5)
    ap.add_argument("--temp", type=float, default=0.8)
    a = ap.parse_args()

    import torch, torchaudio, perth
    if torch.cuda.is_available():
        dev = "cuda"
    elif torch.backends.mps.is_available():
        dev = "mps"
        perth.PerthImplicitWatermarker = perth.DummyWatermarker  # sem backend no Mac
        _load = torch.load
        torch.load = lambda *x, **k: _load(*x, **{**k, "map_location": "cpu"})
    else:
        dev = "cpu"
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    m = ChatterboxMultilingualTTS.from_pretrained(device=dev)

    _, partes = ler_roteiro(Path(a.roteiro))
    pedacos = []
    for i, texto in enumerate(partes):
        print(f"[voz] {i+1}/{len(partes)}: {texto[:60]}…", flush=True)
        w = m.generate(texto, language_id="pt", audio_prompt_path=a.voz_ref,
                       exaggeration=a.exag, cfg_weight=a.cfg, temperature=a.temp)
        pedacos.append(w.cpu())
        pedacos.append(torch.zeros(1, int(m.sr * 0.35)))  # pausa entre frases
    audio = torch.cat(pedacos, dim=1)
    torchaudio.save(a.saida, audio, m.sr)
    print(f"[voz] {audio.shape[-1]/m.sr:.1f}s → {a.saida}")


if __name__ == "__main__":
    main()
''')
sys.path.insert(0, str(TRABALHO))
import importlib, montagem; importlib.reload(montagem)
from montagem import ler_roteiro, montar
VENV_PY = next(p for p in [Path("/content/venv_tts/bin/python"), Path("/kaggle/working/venv_tts/bin/python")] if p.exists())

# referência de voz: aceita .wav ou extrai do .mp4/.mov do briefing
if not voz_ref.exists():
    src = next((p for p in [ENTRADA / "voz_ref.mp4", ENTRADA / "voz_ref.mov"] if p.exists()), None)
    assert src, "coloque voz_ref.wav (ou voz_ref.mp4) na pasta fabrica/"
    voz_ref = TRABALHO / "voz_ref.wav"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(src), "-ac", "1", "-ar", "24000", "-t", "60", str(voz_ref)], check=True)
print("voz_ref:", voz_ref)


In [ ]:
#@title 5. Produzir os reels
import random
PASSOS = 20        # LatentSync: 20 = qualidade cheia
GUIDANCE = 1.5

def lipsync(base: Path, wav16: Path, out: Path):
    cmd = [sys.executable, "-m", "scripts.inference",
           "--unet_config_path", "configs/unet/stage2_512.yaml",
           "--inference_ckpt_path", "checkpoints/latentsync_unet.pt",
           "--inference_steps", str(PASSOS), "--guidance_scale", str(GUIDANCE), "--enable_deepcache",
           "--video_path", str(base), "--audio_path", str(wav16), "--video_out_path", str(out)]
    r = subprocess.run(cmd, cwd=LS, capture_output=True, text=True)
    if r.returncode != 0 or not out.exists():
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError("LatentSync falhou")

for rot in roteiros:
    nome = rot.stem
    pasta = SAIDA / nome
    final = pasta / "reel_final.mp4"
    if final.exists():
        print(f"✓ {nome} já existe, pulando"); continue
    pasta.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); print(f"\n▶ {nome}")
    gancho, _ = ler_roteiro(rot)

    wav = pasta / "voz.wav"
    if not wav.exists():
        r = subprocess.run([str(VENV_PY), str(TRABALHO / "tts.py"), str(rot), str(voz_ref), str(wav)], capture_output=True, text=True)
        if r.returncode != 0: print(r.stderr[-3000:]); raise RuntimeError("voz falhou")
    wav16 = pasta / "voz16.wav"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(wav), "-ar", "16000", "-ac", "1", str(wav16)], check=True)
    print(f"  voz ok ({time.time()-t0:.0f}s)")

    sync = pasta / "sync.mp4"
    if not sync.exists():
        lipsync(random.choice(bases), wav16, sync)
    print(f"  boca ok ({time.time()-t0:.0f}s)")

    montar(sync, wav, gancho, pasta)
    print(f"  ✅ {final} ({time.time()-t0:.0f}s)")

print("\nTODOS PRONTOS →", SAIDA)


In [ ]:
#@title 6. (Kaggle) zipar a saída pra baixar
if NO_KAGGLE:
    shutil.make_archive("/kaggle/working/reels", "zip", SAIDA)
    print("baixe /kaggle/working/reels.zip no painel Output")
else:
    print("Colab: os reels já estão no Drive em", SAIDA)
